# AI5707 — Assignment 6
**Karthikeyan P L | 26120031**

Gaussian mixture data generation, K-Means clustering, distortion-based classification, and covariance comparison.

## 1. Imports and reproducibility
The random seed makes the experiment reproducible.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, ConfusionMatrixDisplay
np.random.seed(42)
plt.rcParams['figure.figsize'] = (7, 5)


## 2. Box–Muller standard normal samples
Each independent pair of uniform random values produces one pair of standard normal values.

In [ ]:
def box_muller(n, rng=np.random):
    u1 = np.clip(rng.uniform(size=n), 1e-12, 1.0)
    u2 = rng.uniform(size=n)
    r = np.sqrt(-2.0 * np.log(u1))
    theta = 2.0 * np.pi * u2
    return np.column_stack((r*np.cos(theta), r*np.sin(theta)))


## 3. Generate two-class Gaussian-mixture data
Each class contains eight Gaussian components. Three covariance structures are supported: spherical, diagonal, and full.

In [ ]:
def make_mixture(class_shift, covariance='full', n_per_component=180, seed=42):
    rng = np.random.default_rng(seed)
    centers = np.array([[-2.4,-1.2],[-2.0,1.4],[-1.1,0.0],[-0.4,1.8],
                        [0.3,-1.6],[0.8,0.7],[1.6,-0.8],[2.0,1.4]]) + np.array([class_shift,0.0])
    if covariance == 'spherical':
        covs = [0.32**2*np.eye(2)]*len(centers)
    elif covariance == 'diagonal':
        covs = [np.diag([0.42**2,0.24**2])]*len(centers)
    else:
        covs = [np.array([[0.38,0.12],[0.12,0.28]])]*len(centers)
    samples=[]
    for mu, cov in zip(centers, covs):
        z = box_muller(n_per_component, rng)
        vals, vecs = np.linalg.eigh(cov)
        transform = vecs @ np.diag(np.sqrt(np.maximum(vals,0)))
        samples.append(z @ transform.T + mu)
    return np.vstack(samples)

X0 = make_mixture(-2.0, 'full', seed=42)
X1 = make_mixture( 2.0, 'full', seed=123)
X = np.vstack((X0, X1))
y = np.r_[np.zeros(len(X0), dtype=int), np.ones(len(X1), dtype=int)]
plt.scatter(X0[:,0], X0[:,1], s=8, alpha=.55, label='Class 1')
plt.scatter(X1[:,0], X1[:,1], s=8, alpha=.55, label='Class 2')
plt.title('Generated Gaussian-mixture data'); plt.legend(); plt.grid(alpha=.2); plt.show()


## 4. Train/test split and K-Means from scratch
Use an 80:20 split. For each class, K-Means learns three centroids (K=3); this K is the number of clusters used by the classifier, not the eight data-generating Gaussian components.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def kmeans(X, k=3, max_iter=100, seed=42):
    rng = np.random.default_rng(seed)
    centers = X[rng.choice(len(X), size=k, replace=False)].copy()
    for _ in range(max_iter):
        labels = np.argmin(((X[:,None,:]-centers[None,:,:])**2).sum(axis=2), axis=1)
        new_centers = np.array([X[labels==j].mean(axis=0) if np.any(labels==j) else centers[j] for j in range(k)])
        if np.allclose(new_centers, centers): break
        centers = new_centers
    return centers

centroids = {c:kmeans(X_train[y_train==c], k=3, seed=100+c) for c in (0,1)}
for c, pts in [(0,'Class 1'),(1,'Class 2')]:
    data = X_train[y_train==c]
    plt.scatter(data[:,0], data[:,1], s=7, alpha=.35, label=pts)
    plt.scatter(centroids[c][:,0], centroids[c][:,1], marker='X', s=150, edgecolor='black', label=f'{pts} centroids')
plt.title('Class-wise K-Means centroids'); plt.legend(); plt.grid(alpha=.2); plt.show()


## 5. Distortion-based classifier and decision boundary
For class c, define distortion as the squared distance to its nearest centroid: `D_c(x) = min_k ||x - μ_ck||²`. Predict the class with the smaller distortion. The decision boundary is the contour where `D_0(x) = D_1(x)`.

In [ ]:
def distortion(X, centers):
    return ((X[:,None,:]-centers[None,:,:])**2).sum(axis=2).min(axis=1)

def predict(X):
    d0, d1 = distortion(X, centroids[0]), distortion(X, centroids[1])
    return (d1 < d0).astype(int)

y_pred = predict(X_test)
print('Test accuracy:', round(accuracy_score(y_test, y_pred), 4))
print('Confusion matrix:\n', confusion_matrix(y_test, y_pred))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['Class 1','Class 2'])
plt.title('Confusion matrix'); plt.show()

x_min,x_max = X[:,0].min()-0.5, X[:,0].max()+0.5
y_min,y_max = X[:,1].min()-0.5, X[:,1].max()+0.5
xx,yy = np.meshgrid(np.linspace(x_min,x_max,350), np.linspace(y_min,y_max,300))
grid = np.c_[xx.ravel(), yy.ravel()]
delta = distortion(grid, centroids[0])-distortion(grid, centroids[1])
plt.scatter(X[y==0,0], X[y==0,1], s=7, alpha=.45, label='Class 1')
plt.scatter(X[y==1,0], X[y==1,1], s=7, alpha=.45, label='Class 2')
plt.contour(xx, yy, delta.reshape(xx.shape), levels=[0], colors='black', linewidths=2)
plt.title('Distortion-based decision boundary'); plt.legend(); plt.grid(alpha=.2); plt.show()


## 6. Compare covariance structures
Repeat data generation for spherical, diagonal, and full covariance. The shape and overlap of components influence how easily the two classes can be separated.

In [ ]:
results = {}
for cov_type in ['spherical','diagonal','full']:
    A = make_mixture(-2.0, cov_type, seed=42)
    B = make_mixture( 2.0, cov_type, seed=123)
    XX = np.vstack((A,B)); yy = np.r_[np.zeros(len(A),dtype=int),np.ones(len(B),dtype=int)]
    Xa,Xb,ya,yb = train_test_split(XX,yy,test_size=.2,random_state=42,stratify=yy)
    cc = {c:kmeans(Xa[ya==c],3,seed=100+c) for c in (0,1)}
    pred = (distortion(Xb,cc[1]) < distortion(Xb,cc[0])).astype(int)
    results[cov_type] = accuracy_score(yb,pred)
    plt.scatter(A[:,0],A[:,1],s=6,alpha=.4,label='Class 1')
    plt.scatter(B[:,0],B[:,1],s=6,alpha=.4,label='Class 2')
    plt.title(f'{cov_type.capitalize()} covariance'); plt.legend(); plt.grid(alpha=.2); plt.show()
print('Test accuracy by covariance structure:')
for name, score in results.items(): print(f'{name:10s}: {score:.4f}')


## 7. Conclusions
- K-Means represents each class using a small set of learned centroids.
- Classification compares the nearest-centroid squared distortion for each class.
- The equal-distortion contour gives the nonlinear decision boundary.
- Covariance structure changes the spread and orientation of mixture components, which can affect classification accuracy.
- Accuracy depends on the random sample, overlap, and the chosen number of centroids.